In [1]:
import torch
import torch.nn.functional as F
from torch import nn
from torch_geometric.nn import SAGEConv, GraphConv, GATConv, to_hetero
from torch_geometric.data import HeteroData
from torch_geometric.transforms import RandomLinkSplit
from sklearn.metrics import roc_auc_score # Common metric for link prediction
import matplotlib.pyplot as plt

import os
import numpy as np
import json
import itertools 
import pandas as pd

from utils.hake_dataset import invert_nested_dict
from utils.utils import *

/home/mai/miniconda3/envs/pyg-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from torch_geometric import seed_everything
seed_everything(4444)

In [3]:
DATA_PATH = './data'

# folder to save entity id mapping
ID_MAPPING = os.path.join(DATA_PATH, 'entity_id_map')
os.makedirs(ID_MAPPING, exist_ok=True)

# folder to save edge index
EDGE_INDEX = os.path.join(DATA_PATH, 'edge_index')
os.makedirs(EDGE_INDEX, exist_ok=True)

# folder to save graph data
GRAPH_DATA = os.path.join(DATA_PATH, 'graph_data')
os.makedirs(GRAPH_DATA, exist_ok=True)

# 0. Load data

## 0.1. Add reverse edge types and localize index for link prediction model

In [4]:
# load data
data = torch.load(os.path.join(GRAPH_DATA, 'yfinance_kge_full.pt'))

id_mapping_path = os.path.join(ID_MAPPING, 'global_id.json')

EDGE_TYPES_ORIGINAL = [
    ('company', 'has_symbol', 'stock_symbol'),
    ('company', 'belongs_to', 'industry'),
    ('industry', 'is_part_of', 'sector'),
    ('institution', 'holds', 'stock_symbol'),
    ('fund', 'holds', 'stock_symbol'),
    ('fund', 'has_symbol', 'fund_symbol'),
]

data, _, FINAL_EDGE_TYPES = localize_and_create_inverse_edges(data, id_mapping_path, EDGE_TYPES_ORIGINAL)

Starting localization of edge indices and creating inverse relations...
✅ Localization complete. Forward edge types: 6, Total edge types: 12


/tmp/ipykernel_37353/4048391829.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(os.path.join(GRAPH_DATA, 'yfinance_kge_full.pt'))


## 0.2. Split the features

In [5]:
HAKE_EMBED_DIM = 32
# Define a minimal constant dimension for nodes that inherently lack fixed features
DEFAULT_FIXED_DIM = 1 
DEFAULT_FIXED_VALUE = 1.0

data = split_and_prepare_features(data, HAKE_EMBED_DIM, DEFAULT_FIXED_DIM, DEFAULT_FIXED_VALUE)

Starting feature split (HAKE D=32, Placeholder D=1)...
  - company: Split features. D_fixed=38.
  - fund_symbol: Split features. D_fixed=29.
  - sector: Split features. D_fixed=7.
  - industry: Split features. D_fixed=9.
  - stock_symbol: Only HAKE found. Assigned default fixed_x (D=1).
  - institution: Only HAKE found. Assigned default fixed_x (D=1).
  - fund: Only HAKE found. Assigned default fixed_x (D=1).
✅ Feature preparation complete.


# 1. Define Feature Manager module

In [6]:
class FeatureManager(nn.Module):
    def __init__(self, data):
        super().__init__()
        
        self.embedding_layers = nn.ModuleDict()
        self.fixed_features = nn.ParameterDict()
        self.node_types = data.node_types
        
        for ntype in self.node_types:
            
            # --- Trainable HAKE Embeddings (nn.Embedding) ---
            if hasattr(data[ntype], 'hake_embeds'):
                hake_tensor = data[ntype].hake_embeds
                N_nodes, D_hake = hake_tensor.shape
                
                # Create trainable nn.Embedding layer, initialized with HAKE weights
                embed_layer = nn.Embedding(N_nodes, D_hake)
                embed_layer.weight.data.copy_(hake_tensor.to(dtype=torch.float))
                self.embedding_layers[ntype] = embed_layer

            # --- Fixed Original Features (nn.Parameter(requires_grad=False)) ---
            if hasattr(data[ntype], 'fixed_x'):
                fixed_tensor = data[ntype].fixed_x
                # Wrap as fixed Parameter, requires_grad=False is CRITICAL here
                fixed_param = nn.Parameter(fixed_tensor.to(dtype=torch.float), requires_grad=False)
                self.fixed_features[ntype] = fixed_param
                
    def forward(self, idx_dict, device):
        """Combines and returns the final feature dictionary for the GNN."""
        x_dict = {}
        for ntype in self.node_types:
            
            fixed_x = self.fixed_features.get(ntype)
            
            # 1. Get trainable HAKE embeddings
            if ntype in self.embedding_layers:
                hake_indices = idx_dict[ntype].to(device)
                trainable_hake_x = self.embedding_layers[ntype](hake_indices)
                
                # 2. Concatenate
                if fixed_x is not None:
                    x_dict[ntype] = torch.cat([fixed_x, trainable_hake_x], dim=-1)
                else:
                    x_dict[ntype] = trainable_hake_x
            
            elif fixed_x is not None:
                 # Case where only fixed features exist (e.g., if D_hake=0 for a node type)
                 x_dict[ntype] = fixed_x
                 
        return x_dict

# 2. Define GNN Encoder

In [7]:
CONV_LAYERS = {
    'SAGE': SAGEConv,
    'GCN': GraphConv, # <-- Correctly using GraphConv for bipartite/hetero graphs
    'GAT': GATConv,
}

class GNNEncoder(nn.Module):
    def __init__(self, 
                 out_channels: int, 
                 num_layers: int = 2, 
                 conv_class: str = 'SAGE', 
                 dropout: float = 0.5,
                 heads: int = 4): 
        """
        Initializes a flexible GNN encoder suitable for wrapping with to_hetero.

        Args:
            out_channels (int): The hidden dimension size.
            num_layers (int): The number of convolutional layers.
            conv_class (str): The type of GNN layer ('SAGE', 'GCN', 'GAT').
            dropout (float): Dropout rate.
            heads (int): Number of attention heads (used only for GAT).
        """
        super().__init__()
        
        self.num_layers = num_layers
        self.act = nn.ReLU()
        self.dropout = nn.Dropout(p=dropout)
        self.heads = heads
        
        ConvLayer = CONV_LAYERS.get(conv_class)
        if ConvLayer is None:
            raise ValueError(f"Unknown convolution type: {conv_class}. Choose from {list(CONV_LAYERS.keys())}")
            
        self.convs = nn.ModuleList()

        # Build the stack of layers
        for i in range(num_layers):
            # All layers (SAGE, GAT, GraphConv) are compatible with the 
            # tuple format (-1, -1) for lazy initialization.
            in_arg = (-1, -1) 
        
            if ConvLayer is GATConv:
                if i < num_layers - 1:
                    # Hidden layers: Use concat=True to combine head outputs. 
                    out_dim = out_channels // heads 
                    conv = GATConv(in_arg, out_dim, heads=heads, concat=True, add_self_loops=False)
                else:
                    # Final layer: Use concat=False to return a single vector of size `out_channels`.
                    conv = GATConv(in_arg, out_channels, heads=heads, concat=False, add_self_loops=False)
            
            else: # SAGEConv or GraphConv (GCN)
                # These layers simply use the full output dimension.
                conv = ConvLayer(in_arg, out_channels,) 

            self.convs.append(conv)

        
    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        """
        Forward pass for the homogeneous GNN.
        """
        
        # Loop through all layers
        for i, conv in enumerate(self.convs):
            
            x = conv(x, edge_index)
            
            # 🛑 CRITICAL FIX: Only apply activation and dropout before the final layer.
            # The final layer's output (embedding) should typically be raw.
            if i < self.num_layers - 1 or self.num_layers == 1: # Apply only to hidden layers
                x = self.act(x)
                x = self.dropout(x)
                
        # The output of the final layer (x) is the node embedding
        return x

# 3. Define Link predictor (decoder)

In [8]:
class EdgeDecoder(torch.nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, z_src, z_dst, edge_label_index):
        # Retrieve the embeddings for the edges' source and destination nodes
        src, dst = edge_label_index
        z_src = z_src[src]
        z_dst = z_dst[dst]
        
        # Calculate the score (e.g., dot product)
        score = torch.sum(z_src * z_dst, dim=-1)
        return score

# 4. Training and eval functions

In [9]:
def split_data(data, num_val, num_test, neg_sampling_ratio, is_undirected, edge_types, rev_edge_types):
    transform = RandomLinkSplit(
        num_val=num_val,
        num_test=num_test,
        neg_sampling_ratio=neg_sampling_ratio,
        is_undirected=is_undirected,
        edge_types=edge_types,
        rev_edge_types=rev_edge_types,
    )
    train_data, val_data, test_data = transform(data)
    return train_data, val_data, test_data

In [10]:
def get_encoder_inputs(data_split, device):
    """
    Prepares the index dictionary required by the FeatureManager.
    
    Args:
        data_split (HeteroData): train_data, val_data, or test_data.
        device (torch.device): The device the tensors should be on.
    
    Returns:
        tuple: (edge_index_dict, idx_dict)
    """
    idx_dict = {}
    
    # Iterate over all node types in the data split
    for ntype in data_split.node_types:
        # Get the index tensor for the FeatureManager's nn.Embedding lookup
        # This attribute (idx) was created during your data preparation step.
        if hasattr(data_split[ntype], 'idx'):
            idx_dict[ntype] = data_split[ntype].idx.to(device)
            
    # Return the graph structure (edge indices) and the node indices
    return data_split.edge_index_dict, idx_dict

In [11]:
def train(train_data, feature_manager, hetero_model, predictor, optimizer, edge_types, criterion, device):
    """
    Performs a single training epoch using the FeatureManager for feature generation.
    """
    feature_manager.train()
    hetero_model.train()
    predictor.train()
    optimizer.zero_grad()
    
    # 1. Prepare inputs (get the graph structure and the indices)
    edge_index_dict, idx_dict = get_encoder_inputs(train_data, device)
    
    # 2. Feature Generation (HAKE lookup and concatenation with fixed features)
    # x_dict is the final combined feature dictionary ready for the GNN
    x_dict = feature_manager(idx_dict, device) 
    
    # 3. GNN Forward Pass
    z_dict = hetero_model(x_dict, edge_index_dict)
    
    total_loss = 0
    
    # 4. Link Prediction and Loss Calculation (using forward EDGE_TYPES only)
    for edge_type in edge_types:
        
        edge_label_index = train_data[edge_type].edge_label_index.to(device)
        edge_labels = train_data[edge_type].edge_label.float().to(device)

        src_type, _, dst_type = edge_type
        
        scores = predictor(
            z_dict[src_type], 
            z_dict[dst_type], 
            edge_label_index
        )

        loss = criterion(scores, edge_labels)
        total_loss += loss
        
    # 5. Backpropagation
    total_loss.backward()
    
    optimizer.step()
    
    return total_loss.item()

In [12]:
@torch.no_grad()
def evaluate(data_split, feature_manager, hetero_model, predictor, edge_types, criterion, device):
    """
    Evaluates the model and returns the total loss and AUC score.
    """
    feature_manager.eval()
    hetero_model.eval()
    predictor.eval()

    # 1. Prepare inputs (get the graph structure and the indices)
    edge_index_dict, idx_dict = get_encoder_inputs(data_split, device)
    
    # 2. Feature Generation
    x_dict = feature_manager(idx_dict, device)
    
    # 3. GNN Forward Pass
    z_dict = hetero_model(x_dict, edge_index_dict)
    
    total_loss = 0
    all_scores = []
    all_labels = []

    # 4. Link Prediction and Metrics Calculation
    for edge_type in edge_types:
        
        edge_label_index = data_split[edge_type].edge_label_index.to(device)
        edge_labels = data_split[edge_type].edge_label.float().to(device)
        
        src_type, _, dst_type = edge_type
        
        scores_logits = predictor(
            z_dict[src_type], 
            z_dict[dst_type], 
            edge_label_index
        )
        
        # Calculate loss
        loss = criterion(scores_logits, edge_labels)
        total_loss += loss.item()

        # Collect scores and labels for AUC
        scores_prob = torch.sigmoid(scores_logits)
        all_scores.append(scores_prob.cpu())
        all_labels.append(edge_labels.cpu())

    # 5. Calculate overall AUC
    try:
        y_score = torch.cat(all_scores).view(-1).numpy()
        y_true = torch.cat(all_labels).view(-1).numpy()

        if len(torch.unique(torch.tensor(y_true))) > 1:
            from sklearn.metrics import roc_auc_score
            auc = roc_auc_score(y_true, y_score)
        else:
            auc = 0.5 
            
    except ValueError:
        auc = 0.5

    return total_loss, auc

In [13]:
import matplotlib.pyplot as plt

def plot_metrics(train_losses, val_losses, val_aucs):
    """
    Plots the training loss, validation loss, and validation AUC over epochs.

    Args:
        train_losses (list): List of training loss values per epoch.
        val_losses (list): List of validation loss values per epoch.
        val_aucs (list): List of validation AUC scores per epoch.
    """
    
    # Calculate the number of epochs from the length of the logged lists
    epochs = range(1, len(train_losses) + 1)
    
    plt.figure(figsize=(12, 5))

    # --- Plot 1: Loss Curve (Train vs. Validation) ---
    plt.subplot(1, 2, 1)
    plt.plot(epochs, train_losses, label='Train Loss', linestyle='-')
    plt.plot(epochs, val_losses, label='Validation Loss', linestyle='--')
    
    plt.title('Loss Curve Over Epochs')
    plt.xlabel('Epoch')
    plt.ylabel('Loss (BCE)')
    plt.legend()
    plt.grid(True, alpha=0.5)

    # --- Plot 2: AUC Curve (Validation Metric) ---
    plt.subplot(1, 2, 2)
    plt.plot(epochs, train_aucs, label='Train AUC', ls='-')
    plt.plot(epochs, val_aucs, label='Validation AUC', ls='-')
    
    plt.title('AUC Score')
    plt.xlabel('Epoch')
    plt.ylabel('AUC Score')
    plt.ylim(0.0, 1.0) # AUC scores are between 0 and 1
    plt.legend()
    plt.grid(True, alpha=0.5)

    plt.tight_layout()
    plt.show()

# 5. Execution

In [14]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

device

device(type='cuda')

In [15]:
# define the search space
EPOCHS = 10000    # maximum epochs to train (early stopping will occur)
PARAM_GRID = {
    'num_layers': [1, 2],
    'conv_type': ['GAT', 'GCN', 'SAGE'],
    'out_channels': [16, 32],
    'learning_rate': [1e-4],
    'num_val': [0.2, 0.3, 0.4],     # train / val / test split (size val = size test)
    
}

In [16]:
NODE_TYPES = ['company', 'stock_symbol', 'industry', 'sector', 
              'institution', 'fund', 'fund_symbol']

In [17]:
# Define PATIENCE_LIMIT here or ensure it's defined globally
PATIENCE_LIMIT = 20 # Wait 20 epochs for an improvement before stopping
LOGGING_FREQUENCY = 200 # Print detailed log every 10 epochs

## 5.1. Link prediction between funds and stock symbols

In [18]:
# only split on the edges we're trying to predict

EDGE_TYPES = [
    # ('institution', 'holds', 'stock_symbol'),
    ('fund', 'holds', 'stock_symbol'),
]

REV_EDGE_TYPES = [
    # ('stock_symbol', 'rev_holds', 'institution'),
    ('stock_symbol', 'rev_holds', 'fund'),
]

EDGE_TYPES_ALL = EDGE_TYPES + REV_EDGE_TYPES

In [19]:
# Prepare the metadata for model initialization
# 1. Extract Shape Information
hake_shapes = {}
fixed_x_shapes = {}
for ntype in data.node_types:
    if hasattr(data[ntype], 'hake_embeds'):
        N, D_hake = data[ntype].hake_embeds.shape
        hake_shapes[ntype] = (N, D_hake)
    if hasattr(data[ntype], 'fixed_x'):
        D_fixed = data[ntype].fixed_x.shape[1]
        fixed_x_shapes[ntype] = D_fixed

# 2. Get the Metadata (contains all node and edge types)
metadata = data.metadata()

In [20]:
# tuning loop
results = []
best_auc = -1.0
best_params = None

# Use itertools.product to get all combinations
keys, values = zip(*PARAM_GRID.items())
param_combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]
num_combinations = len(param_combinations)

print(f"\nStarting hyperparameter search across {num_combinations} configurations...")

for i, params in enumerate(param_combinations):
    
    # --- Configuration Logging ---
    print("\n" + "=" * 60)
    print(f"🔥 Configuration {i+1}/{num_combinations}:")
    print("-" * 60)
    for key, value in params.items():
        print(f"  {key.replace('_', ' ').title():<15}: {value}")
    print("-" * 60)

    # ---- data split -----
    data = data.to(device)
    
    train_data, val_data, test_data = split_data(data, 
                                                 num_val=params['num_val'], 
                                                 num_test=params['num_val'], 
                                                 neg_sampling_ratio=1.0, 
                                                 is_undirected=False, 
                                                 edge_types=EDGE_TYPES,
                                                 rev_edge_types=REV_EDGE_TYPES,
                                                )
    
    # --- 1. Model Initialization ---
    feature_manager = FeatureManager(data).to(device)
    gnn_encoder = GNNEncoder(
        out_channels=params['out_channels'],
        num_layers=params['num_layers'],
        conv_class=params['conv_type'],
        heads=4,
    ).to(device)
    hetero_model = to_hetero(gnn_encoder, metadata, aggr='sum').to(device)
    predictor = EdgeDecoder().to(device)
    optimizer = torch.optim.Adam(
        list(feature_manager.parameters()) + list(hetero_model.parameters()) + list(predictor.parameters()), 
        lr=params['learning_rate']
    )
    criterion = torch.nn.BCEWithLogitsLoss()

    # --- Early Stopping Setup ---
    best_val_auc_for_reporting = -1.0
    best_val_loss = float('inf') # We are tracking loss, so lower is better
    patience_counter = 0
    should_stop = False
    
    # --- 2. Training and Evaluation ---
    
    for epoch in range(1, EPOCHS + 1):
        
        # Train
        current_train_loss = train(train_data, feature_manager, hetero_model, predictor, optimizer, EDGE_TYPES, criterion, device)
        
        # Stop if the early stopping condition has been met
        if should_stop:
            print(f"  Stopping early at epoch {epoch-1} due to loss of patience.")
            break
            
        # Evaluate Validation Set - Run evaluation every epoch for accurate loss tracking
        if epoch % 1 == 0: 
            
            # --- Perform Evaluation ---
            # We evaluate both train and val metrics every epoch for early stopping/logging
            
            # Train evaluation (for logging)
            train_loss_eval, train_auc = evaluate(train_data, feature_manager, hetero_model, predictor, EDGE_TYPES, criterion, device)
            
            # Validation evaluation (for stopping)
            val_loss, val_auc = evaluate(val_data, feature_manager, hetero_model, predictor, EDGE_TYPES, criterion, device)
            
            # --- Early Stopping Logic (Loss-Based) ---
            if val_loss < best_val_loss:
                # Improvement found!
                best_val_loss = val_loss
                best_val_auc_for_reporting = val_auc # Store the AUC that corresponds to the best loss
                patience_counter = 0 # Reset patience
            else:
                patience_counter += 1
                if patience_counter >= PATIENCE_LIMIT:
                    should_stop = True

            # --- Logging ---
            if epoch % LOGGING_FREQUENCY == 0 or epoch == EPOCHS or should_stop:
                 # Ensure to use the current train_loss (returned from train func) or the evaluated one (train_loss_eval)
                 print(f"  Epoch: {epoch:03d}/{EPOCHS}, Train Loss: {train_loss_eval:.4f}, Train AUC: {train_auc:.4f}, Val Loss: {val_loss:.4f}, Val AUC: {val_auc:.4f} (Patience: {patience_counter}/{PATIENCE_LIMIT})")

    # --- 3. Record Results and Track Best Model ---
    
    # Create the result dictionary for the current configuration
    result = params.copy()
    result['final_val_auc'] = best_val_auc_for_reporting # <-- Record the AUC corresponding to the best loss
    result['final_val_loss'] = best_val_loss # <-- Record the best loss
    result['epochs_trained'] = epoch - 1 if should_stop else EPOCHS
    results.append(result)

    # Check if this configuration's BEST AUC is the overall best for the whole tuning loop
    if best_val_auc_for_reporting > best_auc:
        best_auc = best_val_auc_for_reporting
        best_params = params
        print(f"\n*** 🎉 New BEST Model Found! Val AUC: {best_auc:.4f} (from Best Loss) ***")
        print("-" * 60)
    else:
        print("-" * 60)


# --- Final Output and DataFrame Creation ---

# Convert the list of results into a pandas DataFrame
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by='final_val_auc', ascending=False) # Sort for cleaner reporting

print("\n" + "=" * 60)
print("✅ Hyperparameter Search Complete and Results Summary ✅")
print("=" * 60)
print(f"Total Configurations Tested: {num_combinations}")
print(f"Overall Best Validation AUC: {best_auc:.4f}")
print("\nBest Parameters:")
for key, value in best_params.items():
    print(f"  - {key.replace('_', ' ').title()}: {value}")
print("=" * 60)

print("\nDataFrame of all results (`results_df`) created (Top 5):")
print(results_df.head())

print("\n📝 Next Steps:")
print("1. **Analyze `results_df`:** Use the DataFrame to study trends and verify the best configuration.")
print("2. **Final Test:** Proceed with the best parameters found.")


Starting hyperparameter search across 36 configurations...

🔥 Configuration 1/36:
------------------------------------------------------------
  Num Layers     : 1
  Conv Type      : GAT
  Out Channels   : 16
  Learning Rate  : 0.0001
  Num Val        : 0.2
------------------------------------------------------------
  Epoch: 200/10000, Train Loss: 0.6332, Train AUC: 0.7078, Val Loss: 0.6394, Val AUC: 0.6920 (Patience: 0/20)
  Epoch: 400/10000, Train Loss: 0.6031, Train AUC: 0.7894, Val Loss: 0.6114, Val AUC: 0.7718 (Patience: 0/20)
  Epoch: 600/10000, Train Loss: 0.5666, Train AUC: 0.8424, Val Loss: 0.5754, Val AUC: 0.8234 (Patience: 0/20)
  Epoch: 800/10000, Train Loss: 0.5361, Train AUC: 0.8664, Val Loss: 0.5475, Val AUC: 0.8466 (Patience: 0/20)
  Epoch: 1000/10000, Train Loss: 0.5231, Train AUC: 0.8759, Val Loss: 0.5374, Val AUC: 0.8543 (Patience: 0/20)
  Epoch: 1200/10000, Train Loss: 0.5161, Train AUC: 0.8800, Val Loss: 0.5338, Val AUC: 0.8560 (Patience: 0/20)
  Epoch: 1364/1000

In [21]:
results_df.to_csv('./tuning_funds_to_stocksymbol.csv', index=False)

In [30]:
results_df.groupby(['conv_type', 'num_layers'])['final_val_auc'].agg(['mean', 'std'])

mean       std
conv_type num_layers                    
GAT       1           0.862073  0.005950
          2           0.839766  0.008333
GCN       1           0.757917  0.023558
          2           0.640530  0.049952
SAGE      1           0.815557  0.022318
          2           0.881749  0.035746

In [31]:
results_df.groupby(['conv_type', 'num_val'])['final_val_auc'].agg(['mean', 'std'])

mean       std
conv_type num_val                    
GAT       0.2      0.851659  0.012694
          0.3      0.854184  0.015736
          0.4      0.846915  0.015075
GCN       0.2      0.672275  0.084660
          0.3      0.713394  0.070395
          0.4      0.712002  0.072712
SAGE      0.2      0.878453  0.046543
          0.3      0.852274  0.039718
          0.4      0.815231  0.030109

In [32]:
results_df.groupby(['conv_type', 'out_channels'])['final_val_auc'].agg(['mean', 'std'])

mean       std
conv_type out_channels                    
GAT       16            0.850258  0.009754
          32            0.851581  0.017527
GCN       16            0.707383  0.067917
          32            0.691064  0.080905
SAGE      16            0.853164  0.047746
          32            0.844141  0.045566